# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martindiarua/ML_01/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
!git clone https://github.com/martindiarua/ML_01.git
%cd ML_01/data/raw

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")


Cloning into 'ML_01'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 149 (delta 56), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.86 MiB | 12.69 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/ML_01/data/raw


In [ ]:
feature_frame = df[
    [
        "search_volume",
        "competition",
        "ctr",
        "engagement_rate",
        "days_since_last_update"
    ]
].copy()

feature_frame.head()

,search_volume,competition,ctr,engagement_rate,days_since_last_update
0,10.0,0.67,0.76,5.88,20
1,90.0,0.01,0.05,0.00,25
2,0.0,0.00,0.09,0.00,20
3,10.0,0.00,0.49,1.28,22
4,0.0,0.00,0.13,0.00,14


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes

**1. search_volume**
- Meaning: The amount of search demand associated with the content page.
- Missing values: Missing values are present in the dataset and will require handling during preprocessing.
- Categorical: No. This is a numerical feature.
- Available when: It represents historical search information and is available before the refresh-priority decision.

**2. competition**
- Meaning: A measure of the level of competition associated with the page's search terms.
- Missing values: Missing values are present and will require handling during preprocessing.
- Categorical: No. This is a numerical feature.
- Available when: It is available before the refresh-priority decision.

**3. ctr**
- Meaning: The historical click-through rate of the content page.
- Missing values: The dataset provides this field without missing values, so no imputation is currently required for this feature.
- Categorical: No. This is a numerical feature.
- Available when: It is calculated from historical performance and is available before the refresh-priority decision.

**4. engagement_rate**
- Meaning: The historical rate at which users engage with the content page.
- Missing values: The dataset provides this field without missing values, so no imputation is currently required for this feature.
- Categorical: No. This is a numerical feature.
- Available when: It is based on historical engagement and is available before the refresh-priority decision.

**5. days_since_last_update**
- Meaning: The number of days since the content page was last updated.
- Missing values: The dataset provides this field without missing values, so no imputation is currently required for this feature.
- Categorical: No. This is a numerical feature.
- Available when: The age of the existing content is known before deciding whether it should be refreshed.

In [ ]:
print("Feature frame shape:", feature_frame.shape)
print("\nMissing values:")
print(feature_frame.isna().sum())

feature_frame.head()

Feature frame shape: (30000, 5)

Missing values:
search_volume             2468
competition               2468
ctr                          0
engagement_rate              0
days_since_last_update       0
dtype: int64


,search_volume,competition,ctr,engagement_rate,days_since_last_update
0,10.0,0.67,0.76,5.88,20
1,90.0,0.01,0.05,0.00,25
2,0.0,0.00,0.09,0.00,20
3,10.0,0.00,0.49,1.28,22
4,0.0,0.00,0.13,0.00,14


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage Hunt

I tested whether a feature derived directly from the proposed target could artificially improve model performance. I created a proxy `refresh_priority` label for demonstration and then deliberately created `leak_feature` by copying that label.

The leaked feature contains the answer the model is supposed to predict, so using it should produce an unrealistically high score. This demonstrates why label-derived information must not be included as a model feature.

I also treat information that becomes available only after the refresh decision as excluded, because using future information would cause leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

features = [
    "search_volume",
    "competition",
    "ctr",
    "engagement_rate",
    "days_since_last_update"
]

# Create the proxy label for the leakage demonstration
df["refresh_priority"] = (
    (df["ctr"] < 0.05) &
    (df["engagement_rate"] < 0.40)
).astype(int)

# Prepare the data for the honest model
model_df = df[features + ["refresh_priority"]].dropna()

X = model_df[features]
y = model_df["refresh_priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train the honest model
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest Accuracy:", accuracy_score(y_test, pred))


# Deliberately introduce data leakage
model_df["leak_feature"] = model_df["refresh_priority"]

X = model_df[
    [
        "search_volume",
        "competition",
        "days_since_last_update",
        "leak_feature"
    ]
]

y = model_df["refresh_priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train the leaky model
leaky_model = DecisionTreeClassifier(random_state=42)
leaky_model.fit(X_train, y_train)

pred = leaky_model.predict(X_test)

print("Accuracy WITH leakage:", accuracy_score(y_test, pred))


# Remove the leaked feature
model_df.drop(columns=["leak_feature"], inplace=True)

print("Leak feature removed.")

Honest Accuracy: 0.9998184129289994
Accuracy WITH leakage: 1.0
Leak feature removed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### What I Excluded and Why

- **content_id** — Used only to identify individual content pages; it does not describe the page's refresh priority.
- **client_id** — Identifies the client and could cause the model to learn client-specific patterns rather than generalizable content signals.
- **provider_used** — Describes the provider used to produce content and is not necessary for the initial refresh-prioritization task.
- **model_used** — Describes the model used to produce content and is not necessary for the initial refresh-prioritization task.
- **content_type** — Context about the type of content rather than a core performance signal for the initial feature set.
- **main_intent** — Describes search intent and is treated as contextual information rather than one of the five selected features.
- **Any future or label-derived information** — Excluded because it would not be available at the decision moment and could introduce data leakage.
- **refresh_priority** — Excluded as a feature because it is the proposed target/proxy being predicted, not an input to the model.
- **leak_feature** — Deliberately created for the leakage experiment and removed before any honest modeling.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.